# Modeling
Regression baseline and final classification model.

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working from:", ROOT.name)


working from: project


In [2]:
import pandas as pd
from src.modeling import (BASELINE_FEATURES, FACTOR_FEATURES, time_split,
                          fit_classifier, classifier_metrics, fit_regression, regression_metrics)
df = pd.read_csv("data/processed/model_dataset.csv", parse_dates=["date"])
model_df = df.dropna(subset=FACTOR_FEATURES + ["target_excess_next"]).copy()
train, test = time_split(model_df, 0.75)
print(train["date"].min(), train["date"].max())
print(test["date"].min(), test["date"].max())

2017-03-31 00:00:00 2023-05-31 00:00:00
2023-06-30 00:00:00 2025-07-31 00:00:00


In [3]:
reg = fit_regression(train)
regression_metrics(reg, test)

{'rmse': 0.026840457679143656,
 'mae': 0.020738518894425833,
 'r2': -0.47941247449377067}

In [4]:
baseline = fit_classifier(train, BASELINE_FEATURES)
factor = fit_classifier(train, FACTOR_FEATURES)
print("baseline", classifier_metrics(baseline, test, BASELINE_FEATURES))
print("factor augmented", classifier_metrics(factor, test, FACTOR_FEATURES))

baseline {'roc_auc': 0.6833333333333333, 'accuracy': 0.6153846153846154, 'precision': 0.3, 'recall': 0.5, 'f1': 0.375, 'brier': 0.23880728889498024}
factor augmented {'roc_auc': 0.48333333333333334, 'accuracy': 0.5769230769230769, 'precision': 0.2727272727272727, 'recall': 0.5, 'f1': 0.35294117647058826, 'brier': 0.24531010953870222}


Interpretation: the classification model is used as the final risk-ranking model because the regression baseline has weak out-of-sample point-prediction performance. More factor inputs did not improve holdout ROC-AUC.